# Homework 4 Solutions

In [1]:
# This is a code cell that imports the necessary libraries for our session.
import numpy as np                        # NumPy for numerical computations
import scipy as sp                        # SciPy for scientific computing
import sympy as sym                       # SymPy for symbolic mathematics
import matplotlib as mpl                  # Matplotlib for plotting
import matplotlib.pyplot as plt           # Matplotlib pyplot interface
from scipy.integrate import solve_ivp     # ODE solver
mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

This page collects full solutions to select exercises from Homework 4.
Each solution is worked out “by hand” and then, where helpful, confirmed
symbolically using SymPy or other computational tools. See the [Homework
Solutions](https://math341diffeq.netlify.app/select_hw_solutions) page
for the full list.

## Exercise 1 — Interpreting Numerical Output

Consider the initial value problem $x' = -2tx,\ x(0)=1$, which has exact
solution $x(t) = e^{-t^2}$. Euler’s method was used to approximate
$x(1)$ for several step sizes $h$:

| $h$     | Euler approximation $X(1)$ | Error $\lvert x(1)-X(1)\rvert$ |
|---------|----------------------------|--------------------------------|
| $0.2$   | $0.399383$                 | $3.150\times10^{-2}$           |
| $0.1$   | $0.381707$                 | $1.383\times10^{-2}$           |
| $0.05$  | $0.374384$                 | $6.505\times10^{-3}$           |
| $0.025$ | $0.371036$                 | $3.157\times10^{-3}$           |

### (a) Ratio of consecutive errors

In [2]:
hs = [0.2, 0.1, 0.05, 0.025]
errs = [3.150e-2, 1.383e-2, 6.505e-3, 3.157e-3]

for i in range(3):
    ratio = errs[i] / errs[i+1]
    print(f"Error({hs[i]}) / Error({hs[i+1]}) = {ratio:.4f}")

Error(0.2) / Error(0.1) = 2.2777
Error(0.1) / Error(0.05) = 2.1261
Error(0.05) / Error(0.025) = 2.0605

Each ratio is close to $2$. This is exactly what we expect from a method
with **global error $O(h)$**: if $\text{Error}(h) \approx Ch$ for some
constant $C$ (independent of $h$, for $h$ small), then halving $h$
should halve the error: $$
\frac{\text{Error}(h)}{\text{Error}(h/2)} \approx \frac{Ch}{C(h/2)} = 2.
$$ The observed ratios ($2.28, 2.13, 2.06$) are close to, and
approaching, $2$ as $h$ shrinks — consistent with the $O(h)$ theory,
with the discrepancy from exactly $2$ attributable to higher-order terms
that become less significant as $h \to 0$.

### (b) Predicting the error at $h=0.0125$

Since $h=0.0125$ is exactly half of $h=0.025$, and the error should
again roughly halve (per part (a)): $$
\text{Error}(0.0125) \approx \frac{\text{Error}(0.025)}{2} = \frac{3.157\times10^{-3}}{2} \approx 1.58\times10^{-3}.
$$

In [3]:
predicted = errs[3] / 2
print(f"Predicted error at h = 0.0125: {predicted:.3e}")

Predicted error at h = 0.0125: 1.579e-03

> **Tip**
>
> This prediction requires no further computation — it follows directly
> from the observed halving pattern (or equivalently, from the $O(h)$
> global error rate) without running Euler’s method again.

### (c) Step size needed to reduce the error by a factor of 8

Since Euler’s method is first-order accurate,
$\text{Error}(h) \approx Ch$, so error scales **linearly** with $h$. To
reduce the error by a factor of $8$, we need to shrink $h$ by the same
factor of $8$: $$
\frac{\text{Error}(h)}{\text{Error}(h/8)} = \frac{Ch}{C(h/8)} = 8.
$$

$$
\boxed{\text{Shrink } h \text{ by a factor of } 8 \text{ for Euler's method.}}
$$

For the fourth-order Runge–Kutta (RK4) method, the global error is
$O(h^4)$, i.e. $\text{Error}(h) \approx Ch^4$. To reduce the error by a
factor of $8$, we need $$
\frac{\text{Error}(h)}{\text{Error}(h/r)} = \frac{Ch^4}{C(h/r)^4} = r^4 = 8 \quad \Longrightarrow \quad r = 8^{1/4}.
$$

In [4]:
r = 8**(sym.Rational(1,4))
print("Required factor for RK4:", sym.N(r, 5))

Required factor for RK4: 1.6818

$$
\boxed{\text{Shrink } h \text{ by a factor of } 8^{1/4} \approx 1.68 \text{ for RK4.}}
$$

> **Note**
>
> This makes sense intuitively: because RK4’s error already shrinks much
> faster as $h$ decreases (fourth power vs. first power), a much smaller
> reduction in $h$ is needed to achieve the same reduction in error,
> compared to Euler’s method.

------------------------------------------------------------------------

## Section 2.2.2 — Second-Order Linear Homogeneous Equations

### Exercise 1: General Solutions

**(a) $x'' - 4x' + 4x = 0$.**

The characteristic equation is
$\lambda^2 - 4\lambda + 4 = (\lambda-2)^2 = 0$, giving a **repeated
root** $\lambda = 2$. For a repeated root, the general solution is $$
\boxed{x(t) = (C_1 + C_2 t)e^{2t}.}
$$

**(b) $x'' - 2x' = 0$.**

The characteristic equation is
$\lambda^2 - 2\lambda = \lambda(\lambda-2) = 0$, giving **distinct real
roots** $\lambda = 0, 2$. $$
\boxed{x(t) = C_1 + C_2 e^{2t}.}
$$

**(c) $\tfrac12 x'' + x' + \tfrac12 x = 0$.**

Multiplying by $2$: $x'' + 2x' + x = 0$, with characteristic equation
$\lambda^2+2\lambda+1=(\lambda+1)^2=0$, a **repeated root**
$\lambda=-1$. $$
\boxed{x(t) = (C_1 + C_2 t)e^{-t}.}
$$

**(d) $x'' + 4x' + 3x = 0$.**

The characteristic equation is
$\lambda^2+4\lambda+3=(\lambda+1)(\lambda+3)=0$, giving **distinct real
roots** $\lambda=-1,-3$. $$
\boxed{x(t) = C_1 e^{-t} + C_2 e^{-3t}.}
$$

In [5]:
t = sym.symbols('t')
x = sym.Function('x')

for label, eq in [
    ("(a)", sym.Eq(sym.diff(x(t),t,2) - 4*sym.diff(x(t),t) + 4*x(t), 0)),
    ("(b)", sym.Eq(sym.diff(x(t),t,2) - 2*sym.diff(x(t),t), 0)),
    ("(c)", sym.Eq(sym.Rational(1,2)*sym.diff(x(t),t,2) + sym.diff(x(t),t) + sym.Rational(1,2)*x(t), 0)),
    ("(d)", sym.Eq(sym.diff(x(t),t,2) + 4*sym.diff(x(t),t) + 3*x(t), 0)),
]:
    print(label, sym.dsolve(eq, x(t)))

(a) Eq(x(t), (C1 + C2*t)*exp(2*t))
(b) Eq(x(t), C1 + C2*exp(2*t))
(c) Eq(x(t), (C1 + C2*t)*exp(-t))
(d) Eq(x(t), (C1 + C2*exp(-2*t))*exp(-t))

> **Note**
>
> SymPy’s general solutions agree with each boxed formula above (note
> $(C_1 + C_2 e^{-2t})e^{-t} = C_1e^{-t} + C_2e^{-3t}$ for part (d),
> just with the constant relabeled).

### Exercise 2: Initial Value Problems $x(0)=1,\ x'(0)=0$

**(a).** From $x(t) = (C_1+C_2t)e^{2t}$: $x(0)=C_1=1$. Differentiating,
$x'(t) = C_2e^{2t}+2(C_1+C_2t)e^{2t}$, so
$x'(0) = C_2+2C_1 = 0 \Rightarrow C_2=-2$. $$
\boxed{x(t) = (1-2t)e^{2t}.}
$$

**(b).** From $x(t)=C_1+C_2e^{2t}$: $x(0)=C_1+C_2=1$, and
$x'(t)=2C_2e^{2t}$ so $x'(0)=2C_2=0\Rightarrow C_2=0,\ C_1=1$. $$
\boxed{x(t) = 1.}
$$

**(c).** From $x(t)=(C_1+C_2t)e^{-t}$: $x(0)=C_1=1$. Differentiating,
$x'(t)=C_2e^{-t}-(C_1+C_2t)e^{-t}$, so
$x'(0)=C_2-C_1=0\Rightarrow C_2=1$. $$
\boxed{x(t) = (1+t)e^{-t}.}
$$

**(d).** From $x(t)=C_1e^{-t}+C_2e^{-3t}$: $x(0)=C_1+C_2=1$, and
$x'(t)=-C_1e^{-t}-3C_2e^{-3t}$, so
$x'(0)=-C_1-3C_2=0\Rightarrow C_1=-3C_2$. Substituting:
$-3C_2+C_2=1\Rightarrow C_2=-\tfrac12,\ C_1=\tfrac32$. $$
\boxed{x(t) = \frac32 e^{-t} - \frac12 e^{-3t}.}
$$

In [6]:
for label, eq in [
    ("(a)", sym.Eq(sym.diff(x(t),t,2) - 4*sym.diff(x(t),t) + 4*x(t), 0)),
    ("(b)", sym.Eq(sym.diff(x(t),t,2) - 2*sym.diff(x(t),t), 0)),
    ("(c)", sym.Eq(sym.Rational(1,2)*sym.diff(x(t),t,2) + sym.diff(x(t),t) + sym.Rational(1,2)*x(t), 0)),
    ("(d)", sym.Eq(sym.diff(x(t),t,2) + 4*sym.diff(x(t),t) + 3*x(t), 0)),
]:
    sol = sym.dsolve(eq, x(t), ics={x(0): 1, x(t).diff(t).subs(t, 0): 0})
    print(label, sym.simplify(sol))

(a) Eq(x(t), (1 - 2*t)*exp(2*t))
(b) Eq(x(t), 1)
(c) Eq(x(t), (t + 1)*exp(-t))
(d) Eq(x(t), (3*exp(2*t) - 1)*exp(-3*t)/2)

> **Note**
>
> SymPy confirms each particular solution, including the somewhat
> surprising part (b), where the initial conditions collapse the general
> solution to the constant $x(t)=1$.

------------------------------------------------------------------------

## Section 2.2.3 — Second-Order Linear Homogeneous Equations (Complex & Real Roots)

### Exercise 1: General Solutions

For each part, we find the roots of the characteristic equation and
write the corresponding general solution.

**(a) $x''+x'+4x=0$.**
$\lambda = \dfrac{-1\pm\sqrt{1-16}}{2} = -\dfrac12 \pm \dfrac{\sqrt{15}}{2}i$
(complex). $$
\boxed{x(t) = e^{-t/2}\left(C_1\cos\tfrac{\sqrt{15}}{2}t + C_2\sin\tfrac{\sqrt{15}}{2}t\right).}
$$

**(b) $x''-4x'+6x=0$.**
$\lambda = \dfrac{4\pm\sqrt{16-24}}{2} = 2\pm\sqrt2\,i$ (complex). $$
\boxed{x(t) = e^{2t}\left(C_1\cos(\sqrt2\,t) + C_2\sin(\sqrt2\,t)\right).}
$$

**(c) $x''+9x=0$.** $\lambda = \pm 3i$ (purely imaginary). $$
\boxed{x(t) = C_1\cos(3t) + C_2\sin(3t).}
$$

**(d) $x''-12x=0$.** $\lambda = \pm2\sqrt3$ (distinct real). $$
\boxed{x(t) = C_1e^{2\sqrt3\,t} + C_2e^{-2\sqrt3\,t}.}
$$

**(e) $2x''+3x'+3x=0$.**
$\lambda = \dfrac{-3\pm\sqrt{9-24}}{4} = -\dfrac34 \pm \dfrac{\sqrt{15}}{4}i$
(complex). $$
\boxed{x(t) = e^{-3t/4}\left(C_1\cos\tfrac{\sqrt{15}}{4}t + C_2\sin\tfrac{\sqrt{15}}{4}t\right).}
$$

**(f) $\tfrac12x''+\tfrac56x'+\tfrac29x=0$.** Multiplying by $18$:
$9x''+15x'+4x=0$, so
$\lambda = \dfrac{-15\pm\sqrt{225-144}}{18} = \dfrac{-15\pm9}{18}$,
giving $\lambda = -\tfrac13, -\tfrac43$ (distinct real). $$
\boxed{x(t) = C_1e^{-t/3} + C_2e^{-4t/3}.}
$$

In [7]:
for label, eq in [
    ("(a)", sym.Eq(sym.diff(x(t),t,2) + sym.diff(x(t),t) + 4*x(t), 0)),
    ("(b)", sym.Eq(sym.diff(x(t),t,2) - 4*sym.diff(x(t),t) + 6*x(t), 0)),
    ("(c)", sym.Eq(sym.diff(x(t),t,2) + 9*x(t), 0)),
    ("(d)", sym.Eq(sym.diff(x(t),t,2) - 12*x(t), 0)),
    ("(e)", sym.Eq(2*sym.diff(x(t),t,2) + 3*sym.diff(x(t),t) + 3*x(t), 0)),
    ("(f)", sym.Eq(sym.Rational(1,2)*sym.diff(x(t),t,2) + sym.Rational(5,6)*sym.diff(x(t),t) + sym.Rational(2,9)*x(t), 0)),
]:
    print(label, sym.simplify(sym.dsolve(eq, x(t))))

(a) Eq(x(t), (C1*sin(sqrt(15)*t/2) + C2*cos(sqrt(15)*t/2))*exp(-t/2))
(b) Eq(x(t), (C1*sin(sqrt(2)*t) + C2*cos(sqrt(2)*t))*exp(2*t))
(c) Eq(x(t), C1*sin(3*t) + C2*cos(3*t))
(d) Eq(x(t), (C1 + C2*exp(4*sqrt(3)*t))*exp(-2*sqrt(3)*t))
(e) Eq(x(t), (C1*sin(sqrt(15)*t/4) + C2*cos(sqrt(15)*t/4))*exp(-3*t/4))
(f) Eq(x(t), (C1*exp(t) + C2)*exp(-4*t/3))

> **Note**
>
> All six general solutions match. (Part (d)’s solution can equivalently
> be written as $x(t) = A\cosh(2\sqrt3\,t) + B\sinh(2\sqrt3\,t)$.)

### Exercise 2: Initial Value Problems $x(0)=1,\ x'(0)=0$

For each complex-root case, writing
$x(t) = e^{\alpha t}(C_1\cos\beta t + C_2\sin\beta t)$, the conditions
$x(0)=1$ and $x'(0)=0$ give $C_1=1$ and $C_2 = -\alpha/\beta$ (from
$x'(0) = \alpha C_1 + \beta C_2 = 0$).

**(a).** $\alpha=-\tfrac12,\ \beta=\tfrac{\sqrt{15}}{2}$, so
$C_2 = \tfrac{1/2}{\sqrt{15}/2} = \tfrac{1}{\sqrt{15}}$. $$
\boxed{x(t) = e^{-t/2}\left(\cos\tfrac{\sqrt{15}}{2}t + \tfrac{1}{\sqrt{15}}\sin\tfrac{\sqrt{15}}{2}t\right).}
$$

**(b).** $\alpha=2,\ \beta=\sqrt2$, so $C_2 = -2/\sqrt2 = -\sqrt2$. $$
\boxed{x(t) = e^{2t}\left(\cos(\sqrt2\,t) - \sqrt2\sin(\sqrt2\,t)\right).}
$$

**(c).** $\alpha=0,\ \beta=3$, so $C_2=0$. $$
\boxed{x(t) = \cos(3t).}
$$

**(d).** $x(t)=C_1e^{2\sqrt3t}+C_2e^{-2\sqrt3t}$: $x(0)=C_1+C_2=1$,
$x'(0)=2\sqrt3(C_1-C_2)=0\Rightarrow C_1=C_2=\tfrac12$. $$
\boxed{x(t) = \tfrac12\left(e^{2\sqrt3t}+e^{-2\sqrt3t}\right) = \cosh(2\sqrt3\,t).}
$$

**(e).** $\alpha=-\tfrac34,\ \beta=\tfrac{\sqrt{15}}{4}$, so
$C_2 = \tfrac{3/4}{\sqrt{15}/4} = \tfrac{3}{\sqrt{15}} = \tfrac{\sqrt{15}}{5}$.
$$
\boxed{x(t) = e^{-3t/4}\left(\cos\tfrac{\sqrt{15}}{4}t + \tfrac{\sqrt{15}}{5}\sin\tfrac{\sqrt{15}}{4}t\right).}
$$

**(f).** $x(t)=C_1e^{-t/3}+C_2e^{-4t/3}$: $x(0)=C_1+C_2=1$,
$x'(0)=-\tfrac13C_1-\tfrac43C_2=0\Rightarrow C_1=-4C_2$. So
$-4C_2+C_2=1\Rightarrow C_2=-\tfrac13,\ C_1=\tfrac43$. $$
\boxed{x(t) = \tfrac43e^{-t/3} - \tfrac13e^{-4t/3}.}
$$

In [8]:
for label, eq in [
    ("(a)", sym.Eq(sym.diff(x(t),t,2) + sym.diff(x(t),t) + 4*x(t), 0)),
    ("(b)", sym.Eq(sym.diff(x(t),t,2) - 4*sym.diff(x(t),t) + 6*x(t), 0)),
    ("(c)", sym.Eq(sym.diff(x(t),t,2) + 9*x(t), 0)),
    ("(d)", sym.Eq(sym.diff(x(t),t,2) - 12*x(t), 0)),
    ("(e)", sym.Eq(2*sym.diff(x(t),t,2) + 3*sym.diff(x(t),t) + 3*x(t), 0)),
    ("(f)", sym.Eq(sym.Rational(1,2)*sym.diff(x(t),t,2) + sym.Rational(5,6)*sym.diff(x(t),t) + sym.Rational(2,9)*x(t), 0)),
]:
    sol = sym.dsolve(eq, x(t), ics={x(0): 1, x(t).diff(t).subs(t, 0): 0})
    print(label, sym.simplify(sol))

(a) Eq(x(t), (sqrt(15)*sin(sqrt(15)*t/2)/15 + cos(sqrt(15)*t/2))*exp(-t/2))
(b) Eq(x(t), (-sqrt(2)*sin(sqrt(2)*t) + cos(sqrt(2)*t))*exp(2*t))
(c) Eq(x(t), cos(3*t))
(d) Eq(x(t), cosh(2*sqrt(3)*t))
(e) Eq(x(t), (sqrt(15)*sin(sqrt(15)*t/4)/5 + cos(sqrt(15)*t/4))*exp(-3*t/4))
(f) Eq(x(t), (4*exp(t) - 1)*exp(-4*t/3)/3)

> **Note**
>
> All six particular solutions match the boxed formulas above.

------------------------------------------------------------------------

## Section 2.2.4 — Damping and Oscillation

### Exercise 3

For which values of the parameter $k$ (if any) will solutions to
$x''+x'+kx=0$: oscillate with no decay? oscillate with decay? decay
without oscillations?

**By Hand.**

The characteristic equation is $\lambda^2+\lambda+k=0$, with roots $$
\lambda = \frac{-1\pm\sqrt{1-4k}}{2}.
$$

**Case 1: $k > \tfrac14$ (complex roots).** Here $1-4k<0$, so $$
\lambda = -\frac12 \pm \frac{\sqrt{4k-1}}{2}i.
$$ The real part is $-\tfrac12$ for **every** value of $k$ in this range
— it never vanishes. So solutions always decay (since
$\text{Re}(\lambda) = -\tfrac12 < 0$) while oscillating (since the
imaginary part is nonzero). This is **oscillation with decay**, and it
occurs for **every $k > \tfrac14$**.

**Case 2: $0 < k \leq \tfrac14$ (real, negative roots).** Here
$1-4k\ge0$ so both roots are real: $$
\lambda_{1,2} = \frac{-1\pm\sqrt{1-4k}}{2}.
$$ The larger root $\lambda_1 = \dfrac{-1+\sqrt{1-4k}}{2}$ is negative
exactly when $\sqrt{1-4k} < 1$, i.e. when $1-4k<1$, i.e. when $k>0$. So
for $0<k\le\tfrac14$, **both** roots are real and negative, giving
**decay without oscillation** (at $k=\tfrac14$ the roots coincide at
$\lambda=-\tfrac12$, a repeated root, still decaying without
oscillation).

**Case 3: $k \leq 0$.** At $k=0$: roots are $\lambda=0,-1$; the solution
$x(t)=C_1+C_2e^{-t}$ approaches the nonzero constant $C_1$, not decaying
to zero and not oscillating. For $k<0$: $\sqrt{1-4k}>1$, so
$\lambda_1 = \dfrac{-1+\sqrt{1-4k}}{2} > 0$, giving an unstable, growing
(non-decaying) solution.

**Conclusion:**

- **Oscillate with no decay:** *never occurs*, for any real value of
  $k$. Since the coefficient of $x'$ is $1 \neq 0$, the real part of any
  complex root pair is always $-\tfrac12$, which is never zero.
- **Oscillate with decay:** occurs for $\boxed{k > \tfrac14}$.
- **Decay without oscillation:** occurs for
  $\boxed{0 < k \leq \tfrac14}$.

> **Tip**
>
> For $k \le 0$ solutions neither oscillate nor decay to zero (they
> approach a nonzero constant at $k=0$, or grow without bound for $k<0$)
> — worth noting even though the exercise only asks about the three
> listed behaviors.

In [9]:
lam, k = sym.symbols('lambda k', real=True)
char = lam**2 + lam + k
roots = sym.solve(sym.Eq(char, 0), lam)
print("Roots:", roots)
print("Discriminant vanishes (repeated root) at k =", sym.solve(sym.Eq(sym.discriminant(char, lam), 0), k))

for kv in [-0.1, 0, 0.1, 0.2, 0.25, 1]:
    rts = [sym.N(r.subs(k, kv)) for r in sym.solve(sym.Eq(lam**2 + lam + kv, 0), lam)]
    print(f"k={kv}: roots = {rts}")

Roots: [-sqrt(1 - 4*k)/2 - 1/2, sqrt(1 - 4*k)/2 - 1/2]
Discriminant vanishes (repeated root) at k = [1/4]
k=-0.1: roots = [-1.09160797830996, 0.0916079783099616]
k=0: roots = [-1.00000000000000, 0]
k=0.1: roots = [-0.887298334620742, -0.112701665379258]
k=0.2: roots = [-0.723606797749979, -0.276393202250021]
k=0.25: roots = [-0.500000000000000]
k=1: roots = []

> **Note**
>
> The numerical sweep confirms the case analysis: negative real part
> throughout, a repeated real root exactly at $k=1/4$, real roots for
> $k \le 1/4$, and a sign change in the larger real root exactly at
> $k=0$.

------------------------------------------------------------------------

## Section 2.3.1 — Undetermined Coefficients

### Exercise 1: Form of the Trial Guess

For each source term $f(t)$, state the *form* of an initial trial guess
for a particular solution $x_p(t)$ (before checking for duplication with
the homogeneous solution).

**(a) $f(t) = 3t^3-1$** (a degree-3 polynomial). $$
\boxed{x_p(t) = At^3+Bt^2+Ct+D.}
$$

**(c) $f(t) = 12$** (a constant, i.e. a degree-0 polynomial). $$
\boxed{x_p(t) = A.}
$$

**(e) $f(t) = 5\sin 7t$** (a pure sinusoid of frequency $7$). $$
\boxed{x_p(t) = A\cos(7t) + B\sin(7t).}
$$

> **Tip**
>
> The general rule: match the “family” of $f(t)$ — a degree-$n$
> polynomial pairs with a full degree-$n$ polynomial trial guess; a sine
> or cosine of frequency $\omega$ pairs with *both* $\cos\omega t$ and
> $\sin\omega t$ (even if only one appears in $f$), because
> differentiation mixes the two.

### Exercise 2: General Solutions

**(a) $x''+7x = te^{3t}$.**

*Homogeneous:* $\lambda^2+7=0 \Rightarrow \lambda=\pm\sqrt7\,i$, so
$x_h = C_1\cos(\sqrt7t)+C_2\sin(\sqrt7t)$.

*Particular:* Since $f(t)=te^{3t}$ and $3$ is not a root of the
characteristic equation, try $x_p = (At+B)e^{3t}$. Substituting and
matching coefficients (worked below with SymPy) gives $A=\tfrac{1}{16}$,
$B=-\tfrac{3}{128}$.

$$
\boxed{x(t) = C_1\cos(\sqrt7t)+C_2\sin(\sqrt7t) + \left(\frac{t}{16}-\frac{3}{128}\right)e^{3t}.}
$$

**(d) $x''-3x'-4x = 2t^2$.**

*Homogeneous:*
$\lambda^2-3\lambda-4=(\lambda-4)(\lambda+1)=0 \Rightarrow \lambda=4,-1$,
so $x_h = C_1e^{4t}+C_2e^{-t}$.

*Particular:* Since $f(t)=2t^2$ and $0$ is not a root, try
$x_p=At^2+Bt+C$. Substituting:
$2A-3(2At+B)-4(At^2+Bt+C) = -4At^2+(-6A-4B)t+(2A-3B-4C)$. Matching to
$2t^2$: $A=-\tfrac12$, then $B=\tfrac34$, then $C=-\tfrac{13}{16}$.

$$
\boxed{x(t) = C_1e^{4t}+C_2e^{-t} - \frac{t^2}{2}+\frac{3t}{4}-\frac{13}{16}.}
$$

**(h) $x''+x'+2x = t\sin2t$.**

*Homogeneous:*
$\lambda^2+\lambda+2=0 \Rightarrow \lambda=-\tfrac12\pm\tfrac{\sqrt7}{2}i$,
so
$x_h = e^{-t/2}\left(C_1\cos\tfrac{\sqrt7}{2}t+C_2\sin\tfrac{\sqrt7}{2}t\right)$.

*Particular:* Since $f(t) = t\sin2t$ and $2i$ is not a root of the
characteristic equation, try $x_p = (At+B)\cos2t+(Ct+D)\sin2t$.
Substituting and matching coefficients (worked below) gives
$A=C=-\tfrac14,\ B=-\tfrac18,\ D=\tfrac12$.

$$
\boxed{x(t) = e^{-t/2}\left(C_1\cos\tfrac{\sqrt7}{2}t+C_2\sin\tfrac{\sqrt7}{2}t\right) + \left(-\frac{t}{4}-\frac18\right)\cos2t + \left(-\frac{t}{4}+\frac12\right)\sin2t.}
$$

In [10]:
t = sym.symbols('t')
x = sym.Function('x')

for label, eq in [
    ("(a)", sym.Eq(sym.diff(x(t),t,2) + 7*x(t), t*sym.exp(3*t))),
    ("(d)", sym.Eq(sym.diff(x(t),t,2) - 3*sym.diff(x(t),t) - 4*x(t), 2*t**2)),
    ("(h)", sym.Eq(sym.diff(x(t),t,2) + sym.diff(x(t),t) + 2*x(t), t*sym.sin(2*t))),
]:
    print(label, sym.simplify(sym.dsolve(eq, x(t))))

(a) Eq(x(t), C1*sin(sqrt(7)*t) + C2*cos(sqrt(7)*t) + t*exp(3*t)/16 - 3*exp(3*t)/128)
(d) Eq(x(t), C1*exp(-t) + C2*exp(4*t) - t**2/2 + 3*t/4 - 13/16)
(h) Eq(x(t), (C1*sin(sqrt(7)*t/2) + C2*cos(sqrt(7)*t/2) + (-2*sqrt(2)*t*sin(2*t + pi/4) + 4*sin(2*t) - cos(2*t))*exp(t/2)/8)*exp(-t/2))

> **Note**
>
> SymPy’s results agree with the boxed formulas (for part (h), SymPy’s
> particular solution is written in a shifted-phase form; expanding
> $\sin(2t+\pi/4)$ and simplifying shows it is algebraically identical
> to the by-hand result).

------------------------------------------------------------------------

## Section 2.4.2 — Variation of Parameters

### Exercise 1: Particular Solutions via Variation of Parameters

For $x''+p(t)x'+q(t)x=f(t)$ with homogeneous solutions $y_1, y_2$ and
Wronskian $W=y_1y_2'-y_2y_1'$, the variation-of-parameters formula gives
a particular solution $$
x_p(t) = -y_1(t)\int \frac{y_2(t)f(t)}{W(t)}\,dt + y_2(t)\int\frac{y_1(t)f(t)}{W(t)}\,dt.
$$

**(a) $x''+x = \tan t$.**

Homogeneous solutions: $y_1=\cos t,\ y_2=\sin t$, with Wronskian
$W = \cos t\cdot\cos t - \sin t\cdot(-\sin t) = 1$.

$$
x_p = -\cos t\int \sin t\tan t\,dt + \sin t\int\cos t\tan t\,dt.
$$

Since $\cos t\tan t = \sin t$, the second integral is
$\int \sin t\,dt = -\cos t$. Since
$\sin t\tan t = \dfrac{\sin^2t}{\cos t} = \sec t - \cos t$, the first
integral is $\int(\sec t - \cos t)\,dt = \ln|\sec t+\tan t| - \sin t$.
Substituting back: $$
x_p = -\cos t\left(\ln|\sec t+\tan t| - \sin t\right) + \sin t(-\cos t) = -\cos t\ln|\sec t+\tan t| + \sin t\cos t - \sin t\cos t.
$$

The $\sin t\cos t$ terms cancel, leaving

$$
\boxed{x_p(t) = -\cos t\,\ln\left|\sec t+\tan t\right|.}
$$

In [11]:
t = sym.symbols('t', positive=True)

y1, y2 = sym.cos(t), sym.sin(t)
W = y1*sym.diff(y2, t) - y2*sym.diff(y1, t)
f = sym.tan(t)

xp = -y1*sym.integrate(y2*f/W, t) + y2*sym.integrate(y1*f/W, t)
xp_boxed = -sym.cos(t)*sym.log(sym.sec(t) + sym.tan(t))

residual = sym.simplify(sym.diff(xp_boxed, t, 2) + xp_boxed - f)
print("Residual for boxed x_p (should be 0):", residual)

Residual for boxed x_p (should be 0): 0

> **Note**
>
> Direct substitution confirms $x_p(t) = -\cos t\ln|\sec t+\tan t|$
> satisfies $x_p''+x_p = \tan t$ exactly.

**(c) $x''-x = \dfrac1t$.**

Homogeneous solutions: $y_1=e^t,\ y_2=e^{-t}$, with Wronskian
$W = e^t(-e^{-t}) - e^{-t}e^t = -2$.

$$
x_p = -e^t\int e^{-t}\cdot\frac{1}{-2t}\,dt + e^{-t}\int e^t\cdot\frac{1}{-2t}\,dt = \frac12 e^t\int\frac{e^{-t}}{t}\,dt - \frac12 e^{-t}\int\frac{e^t}{t}\,dt.
$$

Neither integral is elementary — they are (up to sign/branch
conventions) the **exponential integral function** $\mathrm{Ei}$: $$
\int\frac{e^t}{t}\,dt = \mathrm{Ei}(t)+C, \qquad \int\frac{e^{-t}}{t}\,dt = \mathrm{Ei}(-t)+C \quad (t>0).
$$

$$
\boxed{x_p(t) = \frac12 e^t\,\mathrm{Ei}(-t) - \frac12 e^{-t}\,\mathrm{Ei}(t).}
$$

In [12]:
from scipy.special import expi

# Numerically verify x_p'' - x_p = 1/t at several points
def xp_c(t0, h=1e-5):
    def f(tt):
        return 0.5*np.exp(tt)*expi(-tt) - 0.5*np.exp(-tt)*expi(tt)
    d2 = (f(t0 + h) - 2*f(t0) + f(t0 - h)) / h**2
    return d2 - f(t0)

for t0 in [0.5, 1.0, 2.0, 3.0]:
    print(f"t={t0}: x_p'' - x_p = {xp_c(t0):.6f}, expected 1/t = {1/t0:.6f}")

t=0.5: x_p'' - x_p = 2.000003, expected 1/t = 2.000000
t=1.0: x_p'' - x_p = 1.000007, expected 1/t = 1.000000
t=2.0: x_p'' - x_p = 0.500002, expected 1/t = 0.500000
t=3.0: x_p'' - x_p = 0.333333, expected 1/t = 0.333333

> **Note**
>
> The residual $x_p''-x_p$ matches $1/t$ closely at every test point
> (small differences are finite-difference truncation error, not a flaw
> in the formula).

**(e) $x''+x = \dfrac{1}{t+1}$.**

Homogeneous solutions and Wronskian are the same as in part (a):
$y_1=\cos t,\ y_2=\sin t,\ W=1$. Substituting $u=t+1$ (so $t=u-1$) and
using the angle-subtraction formulas, the integrals become the **sine
and cosine integral functions** $\mathrm{Si}, \mathrm{Ci}$: $$
\int\frac{\sin t}{t+1}\,dt = \cos(1)\,\mathrm{Si}(t+1) - \sin(1)\,\mathrm{Ci}(t+1), \qquad
\int\frac{\cos t}{t+1}\,dt = \cos(1)\,\mathrm{Ci}(t+1) + \sin(1)\,\mathrm{Si}(t+1).
$$

Therefore

$$
\boxed{x_p(t) = -\cos t\Big[\cos(1)\mathrm{Si}(t+1) - \sin(1)\mathrm{Ci}(t+1)\Big] + \sin t\Big[\cos(1)\mathrm{Ci}(t+1) + \sin(1)\mathrm{Si}(t+1)\Big].}
$$

In [13]:
from scipy.special import sici

c1, s1 = np.cos(1), np.sin(1)

def xp_e(t0, h=1e-5):
    def f(tt):
        Si_u, Ci_u = sici(tt + 1)
        return -np.cos(tt)*(c1*Si_u - s1*Ci_u) + np.sin(tt)*(c1*Ci_u + s1*Si_u)
    d2 = (f(t0 + h) - 2*f(t0) + f(t0 - h)) / h**2
    return d2 + f(t0)

for t0 in [0.0, 0.5, 1.0, 2.0]:
    print(f"t={t0}: x_p'' + x_p = {xp_e(t0):.6f}, expected 1/(t+1) = {1/(t0+1):.6f}")

t=0.0: x_p'' + x_p = 1.000000, expected 1/(t+1) = 1.000000
t=0.5: x_p'' + x_p = 0.666663, expected 1/(t+1) = 0.666667
t=1.0: x_p'' + x_p = 0.500003, expected 1/(t+1) = 0.500000
t=2.0: x_p'' + x_p = 0.333332, expected 1/(t+1) = 0.333333

> **Note**
>
> Again, direct numerical substitution confirms the closed-form
> particular solution — a good check when an integral doesn’t reduce to
> elementary functions.

> **Tip**
>
> Parts (c) and (e) illustrate an important point: the variation of
> parameters formula *always* produces a valid particular solution in
> principle, but the resulting integrals are not always expressible in
> terms of elementary functions. When that happens, it’s standard
> practice to express the answer using well-studied special functions
> (here, $\mathrm{Ei}$, $\mathrm{Si}$, $\mathrm{Ci}$) rather than
> leaving the answer as an unevaluated integral.

## References

> **Expand for Session Info**
>
> ``` python
> import sys # sys for system-specific parameters and functions
> print("Python version:", sys.version)
> print('\n'.join(f'{m.__name__}=={m.__version__}' for m in globals().values() if getattr(m, '__version__', None)))
> ```
>
>     Python version: 3.14.4 | packaged by conda-forge | (main, Apr  8 2026, 02:33:53) [Clang 20.1.8 ]
>     numpy==2.4.3
>     scipy==1.17.1
>     sympy==1.14.0
>     matplotlib==3.10.8

## Reuse

[![](http://mirrors.creativecommons.org/presskit/buttons/88x31/png/by-nc-sa.png?raw=1)](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode)

[CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)